# Sesgos algorítmicos introducidos por sample_weight 
Veamos dos ejemplos en que los datos son "ideales" (balanceados, misma prevalencia y calidad por grupo), pero las decisiones de entrenamiento con sample_weight introducen sesgos algorítmicos.

In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [4]:
# Función de ayuda para evaluar métricas por grupo
def evaluate_by_group(model, X_test, y_test, groups, group_names):
    y_pred = model.predict(X_test)
    print("Métricas globales:")
    print(classification_report(y_test, y_pred, digits=3))
    print("\nMétricas por grupo:")
    for g in np.unique(groups):
        idx = groups == g
        print(f"--- Grupo {group_names[g]} ---")
        print(classification_report(y_test[idx], y_pred[idx], digits=3))

# Ejemplo 1: Fraude en pagos online (País A vs País B)

# Datos sintéticos balanceados: 5000 muestras, 2 países (0=A, 1=B)
X, y = make_classification(n_samples=5000, n_features=10, n_informative=5, 
                           weights=[0.95, 0.05], random_state=42)

# Agregar variable de país (balanceada 50/50)
np.random.seed(42)
groups = np.random.choice([0,1], size=len(y))  # 0=A, 1=B
group_names = {0:"País A", 1:"País B"}

X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X, y, groups, test_size=0.3, stratify=y, random_state=42)

# Baseline sin pesos
clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)
print("=== Baseline sin pesos ===")
evaluate_by_group(clf, X_test, y_test, g_test, group_names)

# Con pesos: damos más importancia al País A
weights = np.where(g_train==0, 10, 1) #(condicion, valor si verdadero, valor si falso)
clf_w = LogisticRegression(max_iter=200)
clf_w.fit(X_train, y_train, sample_weight=weights)
print("\n=== Con sample_weight favoreciendo País A ===")
evaluate_by_group(clf_w, X_test, y_test, g_test, group_names)



=== Baseline sin pesos ===
Métricas globales:
              precision    recall  f1-score   support

           0      0.950     0.998     0.974      1418
           1      0.727     0.098     0.172        82

    accuracy                          0.949      1500
   macro avg      0.839     0.548     0.573      1500
weighted avg      0.938     0.949     0.930      1500


Métricas por grupo:
--- Grupo País A ---
              precision    recall  f1-score   support

           0      0.952     0.996     0.973       710
           1      0.500     0.077     0.133        39

    accuracy                          0.948       749
   macro avg      0.726     0.536     0.553       749
weighted avg      0.928     0.948     0.929       749

--- Grupo País B ---
              precision    recall  f1-score   support

           0      0.949     1.000     0.974       708
           1      1.000     0.116     0.208        43

    accuracy                          0.949       751
   macro avg      0

In [11]:
# Ejemplo 2: Salud pública (Joven vs Anciano)

# Datos sintéticos balanceados: 4000 muestras
X2, y2 = make_classification(n_samples=4000, n_features=8, n_informative=4,
                             weights=[0.92, 0.08], random_state=0)

# Variable de sexo (0=Joven, 1=Anciano), balanceada
np.random.seed(0)
groups2 = np.random.choice([0,1], size=len(y2))
group_names2 = {0:"Joven", 1:"Anciano"}

X2_train, X2_test, y2_train, y2_test, g2_train, g2_test = train_test_split(
    X2, y2, groups2, test_size=0.3, stratify=y2, random_state=0)

# Baseline sin pesos
clf2 = LogisticRegression(max_iter=200)
clf2.fit(X2_train, y2_train)
print("\n=== Baseline Salud (sin pesos) ===")
evaluate_by_group(clf2, X2_test, y2_test, g2_test, group_names2)

# Con pesos favoreciendo a Anciano (por mayor riesgo clínico)
# los errores cometidos en las muestras con peso 10 (clase 1) penalizan cinco veces más que los errores en las muestras con peso 1 (otras clases).
weights2 = np.where(g2_train==1, 10, 1)
clf2_w = LogisticRegression(max_iter=200)
clf2_w.fit(X2_train, y2_train, sample_weight=weights2)
print("\n=== Con sample_weight favoreciendo Anciano ===")
evaluate_by_group(clf2_w, X2_test, y2_test, g2_test, group_names2)


=== Baseline Salud (sin pesos) ===
Métricas globales:
              precision    recall  f1-score   support

           0      0.938     1.000     0.968      1098
           1      1.000     0.284     0.443       102

    accuracy                          0.939      1200
   macro avg      0.969     0.642     0.705      1200
weighted avg      0.943     0.939     0.923      1200


Métricas por grupo:
--- Grupo Joven ---
              precision    recall  f1-score   support

           0      0.937     1.000     0.968       536
           1      1.000     0.265     0.419        49

    accuracy                          0.938       585
   macro avg      0.969     0.633     0.693       585
weighted avg      0.942     0.938     0.922       585

--- Grupo Anciano ---
              precision    recall  f1-score   support

           0      0.938     1.000     0.968       562
           1      1.000     0.302     0.464        53

    accuracy                          0.940       615
   macro a

## Conclusión
En ambos casos los datos eran "ideales" (balanceados, misma prevalencia, misma calidad).
Sin embargo, al introducir sample_weight para reflejar decisiones de negocio o de salud,el modelo optimiza mejor para un grupo que para otro. Esto genera un sesgo algorítmico:
- En fraude: se favorece recall en País A, a costa de País B.
- En salud: se favorece recall en Anciano, a costa de Joven.